In [1]:
from util import *

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
import pickle

from scipy import stats
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, make_scorer
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.decomposition import PCA
from joblib import dump, load

In [2]:
data_log, data_dash = get_data_frames()
result_total, labels = get_data_concated(data_log, data_dash)

In [3]:
def nmae(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    nmae_value = mae / np.mean(y_true)
    return nmae_value
def train_rf_model_gridsearch(features_train: pd.DataFrame, labels_train: pd.Series):
    X_train, X_validation, y_train, y_validation = train_test_split(
        features_train,
        np.ravel(labels_train),
        test_size=0.20,
        random_state=42
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_validation_scaled = scaler.transform(X_validation)

    rf_model = RandomForestRegressor(random_state=42, n_jobs=2)

    nmae_scorer = make_scorer(nmae, greater_is_better=False)

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    n_estimators = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150]
    max_features = ['auto', 'sqrt']
    max_depth = [int(x) for x in np.linspace(start=10, stop=100, num=11)]
    max_depth.append(None)
    min_samples_split = [2, 5, 10]
    min_samples_leaf = [1, 2, 4]
    bootstrap = [True, False]

    random_grid = {
        'n_estimators': n_estimators,
        'max_features': max_features,
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf,
        'bootstrap': bootstrap
    }

    grid_search = RandomizedSearchCV(
        estimator=rf_model,
        param_distributions=random_grid,
        scoring=nmae_scorer,
        cv=kf,
        n_iter=100, 
        verbose=3,  
        n_jobs=2
    )

    grid_search.fit(X_train_scaled, y_train)

    best_rf_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    print(f"Melhores hiperparâmetros: {best_params}")

    y_pred_rf = best_rf_model.predict(X_validation_scaled)
    mae_rf = mean_absolute_error(y_validation, y_pred_rf)
    nmae_rf = nmae(y_validation, y_pred_rf)

    feature_importances = best_rf_model.feature_importances_
    print(f"Importâncias das features: {feature_importances}")

    return mae_rf, nmae_rf, best_rf_model, feature_importances


In [4]:
mae, nmae, model, pred = train_rf_model_gridsearch(result_total.copy(deep=True), labels)
print(f"{nmae*100:.4f}%")
alert_end()

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV 1/5] END bootstrap=True, max_depth=73, max_features=auto, min_samples_leaf=2, min_samples_split=10, n_estimators=80;, score=nan total time=   0.0s
[CV 2/5] END bootstrap=True, max_depth=73, max_features=auto, min_samples_leaf=2, min_samples_split=10, n_estimators=80;, score=nan total time=   0.0s
[CV 3/5] END bootstrap=True, max_depth=73, max_features=auto, min_samples_leaf=2, min_samples_split=10, n_estimators=80;, score=nan total time=   0.0s
[CV 4/5] END bootstrap=True, max_depth=73, max_features=auto, min_samples_leaf=2, min_samples_split=10, n_estimators=80;, score=nan total time=   0.0s
[CV 5/5] END bootstrap=True, max_depth=73, max_features=auto, min_samples_leaf=2, min_samples_split=10, n_estimators=80;, score=nan total time=   0.0s
[CV 1/5] END bootstrap=False, max_depth=91, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=80;, score=-0.293 total time=   7.8s
[CV 2/5] END bootstrap=False

KeyboardInterrupt: 